# Customer Support Ticket Classification

**Tech stack:** Python, Pandas, NumPy, Scikit-learn, Matplotlib, Jupyter Notebook

This self-contained notebook cleans and analyzes customer-support tickets, builds leakage-safe TF-IDF pipelines, compares three classifiers, evaluates the selected model, and explains its predictions.

> **Data disclosure:** `synthetic_support_tickets.csv` is reproducibly generated development data—not real customer traffic. It includes 15% controlled routing-label ambiguity to avoid an unrealistically perfect benchmark. Results demonstrate the workflow and must not be presented as production validation.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay,
                             precision_recall_fscore_support)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Load the dataset

In [ ]:
DATA_PATH = Path('synthetic_support_tickets.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent / 'synthetic_support_tickets.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError('Place synthetic_support_tickets.csv beside this notebook.')

raw = pd.read_csv(DATA_PATH)
print(f'Rows: {len(raw):,} | Columns: {raw.shape[1]}')
raw.head(3)

## 3. Data inspection and quality checks

In [ ]:
display(pd.DataFrame({'dtype': raw.dtypes.astype(str), 'missing': raw.isna().sum()}))
print('Exact duplicates:', raw.duplicated().sum())
print('Blank subjects:', raw['subject'].fillna('').str.strip().eq('').sum())
print('Blank descriptions:', raw['description'].fillna('').str.strip().eq('').sum())

## 4. Cleaning and text preparation

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()

df = raw.drop_duplicates().copy()
for column in ['subject', 'description', 'category']:
    df[column] = df[column].map(normalize_text)
df = df[(df['category'] != '') & ((df['subject'] != '') | (df['description'] != ''))].copy()
df['text'] = (df['subject'] + ' ' + df['description']).map(normalize_text)
df['text_length'] = df['text'].str.len()
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
print(f'Clean rows: {len(df):,} | Removed: {len(raw)-len(df):,}')

## 5. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
df['category'].value_counts().sort_values().plot.barh(ax=axes[0,0], color='#5965d8', title='Ticket category distribution')
df['priority'].value_counts().plot.bar(ax=axes[0,1], color='#25a77b', title='Priority distribution', rot=0)
df['channel'].value_counts().plot.bar(ax=axes[1,0], color='#ef9f43', title='Channel distribution', rot=0)
df['text_length'].plot.hist(ax=axes[1,1], bins=25, color='#7b61a8', title='Ticket text length')
axes[1,1].set_xlabel('Characters')
plt.tight_layout(); plt.show()

## 6. Operational analysis: resolution and satisfaction

In [ ]:
operational = df.groupby('category').agg(
    tickets=('ticket_id','count'),
    median_resolution_hours=('resolution_time_hours','median'),
    average_satisfaction=('customer_satisfaction','mean')
).sort_values('tickets', ascending=False).round(2)
display(operational)
display(pd.crosstab(df['category'], df['priority'], normalize='index').round(3))

monthly = df.set_index('created_at').resample('ME')['ticket_id'].count()
monthly.plot(figsize=(11,4), color='#5965d8', title='Monthly ticket volume')
plt.ylabel('Tickets'); plt.show()

## 7. Leakage-safe train/test split and TF-IDF pipeline

Only subject and description are used. TF-IDF is fitted inside each pipeline after the stratified split, preventing test-set leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['category'], test_size=0.20,
    random_state=RANDOM_STATE, stratify=df['category'])

def make_pipeline(model):
    return Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english',
                                  ngram_range=(1,2), min_df=2,
                                  max_df=0.98, max_features=12000)),
        ('classifier', model)
    ])

print('Training rows:', len(X_train), '| Test rows:', len(X_test))

## 8. Train and compare models

In [ ]:
candidates = {
    'Logistic Regression': LogisticRegression(max_iter=1200, class_weight='balanced', random_state=RANDOM_STATE),
    'LinearSVC': LinearSVC(class_weight='balanced', random_state=RANDOM_STATE),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.3),
}

trained, rows = {}, []
for name, estimator in candidates.items():
    model = make_pipeline(estimator).fit(X_train, y_train)
    prediction = model.predict(X_test)
    p, r, f_macro, _ = precision_recall_fscore_support(y_test, prediction, average='macro', zero_division=0)
    _, _, f_weighted, _ = precision_recall_fscore_support(y_test, prediction, average='weighted', zero_division=0)
    rows.append({'Model': name, 'Accuracy': accuracy_score(y_test, prediction),
                 'Macro Precision': p, 'Macro Recall': r,
                 'Macro F1': f_macro, 'Weighted F1': f_weighted})
    trained[name] = model

comparison = pd.DataFrame(rows).sort_values(['Macro F1','Accuracy'], ascending=False).reset_index(drop=True)
display(comparison.style.format({column:'{:.3f}' for column in comparison.columns if column != 'Model'}))

## 9. Selected model and detailed evaluation

In [ ]:
best_name = comparison.loc[0, 'Model']
best_model = trained[best_name]
y_pred = best_model.predict(X_test)
best_accuracy = accuracy_score(y_test, y_pred)
_, _, best_macro_f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
_, _, best_weighted_f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
print(f'Selected model: {best_name}')
print(f'Accuracy: {best_accuracy:.3%}')
print(f'Macro F1: {best_macro_f1:.3f} | Weighted F1: {best_weighted_f1:.3f}')
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, xticks_rotation=35, cmap='Blues', colorbar=False)
plt.title(f'Confusion matrix — {best_name}')
plt.tight_layout(); plt.show()

## 10. Explainability: influential terms

In [ ]:
explain_model = trained['Logistic Regression']
terms = explain_model.named_steps['tfidf'].get_feature_names_out()
classifier = explain_model.named_steps['classifier']
for label, coefficients in zip(classifier.classes_, classifier.coef_):
    top = terms[np.argsort(coefficients)[-8:][::-1]]
    print(f'{label:20s}: {", ".join(top)}')

## 11. Error analysis

In [ ]:
errors = pd.DataFrame({'text': X_test, 'actual': y_test, 'predicted': y_pred})
errors = errors[errors['actual'] != errors['predicted']]
print(f'Misclassified tickets: {len(errors)} of {len(y_test)}')
errors.head(10)

## 12. Sample predictions

In [ ]:
samples = pd.Series([
    'I reset my password but my account remains locked',
    'The tracking page says the parcel went to the wrong address',
    'My card was charged twice for the same invoice'
])
pd.DataFrame({'ticket': samples, 'predicted_category': best_model.predict(samples)})

## 13. Key findings

- Ticket categories are balanced by construction, so macro and weighted metrics are comparable.
- Category-specific words remain the strongest routing signals.
- Controlled label ambiguity produces meaningful errors for inspection instead of an unrealistically perfect score.
- Resolution time and satisfaction fields support operational analysis but are excluded from classification to avoid using information unavailable at ticket creation.

## 14. Limitations and conclusion

This workflow is reproducible and demonstrates cleaning, operational EDA, leakage-safe feature engineering, model comparison, evaluation, explainability, and prediction. However, the data is synthetic and shares generator vocabulary across splits. The measured score must be described as a **synthetic benchmark**, not expected production accuracy. A production system should retrain on representative consented tickets, use temporal/out-of-domain validation, calibrate confidence, monitor drift, and retain human review for uncertain cases.